# 🎬 FilmChain Studio - Google Colab 雲端無縫長影片生成器

歡迎使用 **FilmChain 雲端生成器**！專為 Google Gemini Pro 用戶量身打造。

本筆記本會全自動：
1. 依據您輸入的 `SESSION_ID`，自動連線 [FilmChain Studio](https://film-chain.onlytest.dev) 拉取分鏡清單、提示詞與高清九宮格圖。
2. 指派 Colab GPU (預設 A100) 進行連鎖推論，自動將前一段最後一幀無縫傳遞給下一段作為首幀。
3. 實時回報進度至網頁端與手機端（呼吸卡片、進度條、即時日誌）。
4. 透過 FFmpeg 精確消除首格重複幀（0.04167s），無縫拼接成單一長影片。
5. 影片自動上傳至雲端中繼站，讓您在網頁播放廳與手機上即時觀看回放並下載！

In [ ]:
# @title 🚀 1. 設定 Session ID 與運算規格 { run: "auto" }
# @markdown 請貼上您在 [FilmChain Studio](https://film-chain.onlytest.dev) 複製的 8 碼 Session ID：
SESSION_ID = "貼上你的 Session ID"  # @param {type:"string"}
GPU_TYPE = "A100"  # @param ["A100", "L4", "T4"]
RELAY_URL = "https://film-chain.onlytest.dev"  # @param {type:"string"}

session_id = SESSION_ID.strip()
if not session_id or session_id == "貼上你的 Session ID":
    print("⚠️ 請在上方輸入框貼上您的 Session ID (例如: fc-abcdef) 後再點擊執行！")
else:
    print(f"✅ 已設定 Session ID: {session_id}")
    print(f"🎯 中繼站位址: {RELAY_URL}")
    print(f"⚡ 目標 GPU: {GPU_TYPE}")

In [ ]:
# @title ⚙️ 2. 初始化環境與下載 FilmChain 工具鏈
import os, sys, shutil, subprocess
from pathlib import Path

print("📦 正在下載最新 FilmChain 專案...")
!git clone --depth 1 https://github.com/yshonlytest/film-chain.git /content/film-chain 2>/dev/null || (cd /content/film-chain && git pull)
%cd /content/film-chain

print("🔧 正在確認 colab CLI 與 FFmpeg...")
!pip install -q google-colab-cli
!apt-get update -qq && apt-get install -y -qq ffmpeg
print("✨ 環境建置完成！")

In [ ]:
# @title 🔑 3. 檢查 Colab OAuth 授權與計算單元餘額
import subprocess, json

print("🔍 正在檢查 Colab 認證餘額...")
res = subprocess.run([sys.executable, "skills/minimax-h3-colab/scripts/runner.py", "usage", "--json"], capture_output=True, text=True)
try:
    usage = json.loads(res.stdout)
    if usage.get("ok"):
        print(f"🎉 認證成功！目前可用餘額: {usage['balance']:.2f} Compute Units (每小時約消耗 {usage['rate_per_hour']:.2f} Units)")
    else:
        print(f"⚠️ 檢測回傳: {usage}")
except Exception:
    print("💡 若尚未授權，請執行下方指令登入：")
    print("!colab login")

In [ ]:
# @title 🎬 4. 一鍵全自動雲端連鎖生成、裁幀縫合與直傳回放
# @markdown 若在網頁點擊了「重抽」，可直接執行，系統會自動依雲端任務僅重跑該段；亦可手動指定：
REROLL_PART = 0  # @param {type:"integer"} # 0 為依雲端任務自動判斷，1~N 為強制指定重抽第 N 段
CASCADE = False  # @param {type:"boolean"} # 重抽時是否連鎖重跑後續段落

import sys, subprocess

session_id = SESSION_ID.strip()
if not session_id or session_id == "貼上你的 Session ID":
    raise ValueError("❌ 請在步驟 1 輸入框填入您的 Session ID！")

print(f"🚀 啟動連鎖生成佇列 (Session: {session_id})...")
print("📱 請保持本分頁開啟，您亦可隨時在網頁或手機端監看呼吸卡片動態！")

cmd = [
    sys.executable,
    "skills/minimax-h3-colab/scripts/runner.py",
    "chain",
    "--session-id", session_id,
    "--relay-url", RELAY_URL,
    "--gpu", GPU_TYPE,
    "--output-dir", "/content/output"
]
if REROLL_PART > 0:
    cmd.extend(["--reroll", str(REROLL_PART)])
    if CASCADE:
        cmd.append("--cascade")

process = subprocess.run(cmd)
if process.returncode == 0:
    print("\n=======================================================")
    print("🎉 恭喜！分鏡渲染、FFmpeg 無縫去幀縫合已就緒，並已直傳至雲端！")
    print(f"👉 請回到網頁控制台或手機端直接播放影片：{RELAY_URL}")
    print("=======================================================")
else:
    print("\n❌ 生成過程遭遇中斷或錯誤，請檢查上方輸出日誌或於網頁端重試。")